# 07 — Revision Tables  [R1-7, R1-8, R1-9, R2-6]

Reproduces the tables that the reviewer revision added or expanded:
**Table 7** (estimator configuration and convergence), **Table 9** (corrected equivalence with TOST p), **Table 10** (Cohen's d_z), **Table 13** (model-class sensitivity, all four configurations), and **Table 15** (the no-UNITSTOTAL analysis in full parallel with the main one).

In [1]:
import json, inspect, warnings, os, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import (StandardScaler, RobustScaler, PowerTransformer, MinMaxScaler, QuantileTransformer)
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score, recall_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from scipy import stats
from lightgbm import LGBMClassifier
os.makedirs("results", exist_ok=True)
RS = 42
TEAL,ROSE,ORANGE,SAND,RED,LAV,SLATE,INK = ('#4A90A4','#C4887D','#E67E22','#C4A882','#C0392B','#8E7CC3','#7A8B9A','#2C3E4F')
plt.rcParams.update({'font.family':'DejaVu Sans','axes.facecolor':'#FAFBFC','figure.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.color':'#E4E9F0',
    'axes.titleweight':'bold','axes.axisbelow':True,'savefig.dpi':200,'savefig.bbox':'tight'})
NAMES = ['Manual Expert','DeepSeek','Claude Sonnet 4.6','GPT-5.4','CAAFE']
SHORT = ['Manual','DeepSeek','Claude 4.6','GPT-5.4','CAAFE']
COLS  = [TEAL,ROSE,RED,ORANGE,LAV]
CONT  = ['AGE','ANNUALCONTRIBUTION','ANNUALCLAIMAMOUNT','UNITSTOTAL']
DATA  = "Final Prepared Dataset - Diabetes and Hypertension Data.xlsx"

def load_raw():
    df = pd.read_excel(DATA)
    n_raw = len(df); df = df.drop_duplicates().reset_index(drop=True); n_dup = n_raw - len(df)
    df["ADHERENCE"] = (df["ADHERENCE"]=="ADHERENT").astype(int)
    for c in df.columns:
        if df[c].dtype=="bool": df[c]=df[c].astype(int)
    return df, n_raw, n_dup

def load_and_split():
    df,_,_ = load_raw()
    y=df["ADHERENCE"]; X=df.drop(columns=["ADHERENCE"])
    return train_test_split(X,y,test_size=0.2,stratify=y,random_state=RS)

def _manual(Xp):
    Xp=Xp.copy()
    Xp["LOG_CLAIM"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]); Xp["LOG_UNITS"]=np.log1p(Xp["UNITSTOTAL"])
    Xp["LOG_CONTRIB"]=np.log1p(Xp["ANNUALCONTRIBUTION"])
    Xp["CLAIM_PER_UNIT"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]/(Xp["UNITSTOTAL"]+1))
    Xp["CLAIM_RATIO"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]/(Xp["ANNUALCONTRIBUTION"]+1))
    Xp["UNITS_PER_AGE"]=Xp["UNITSTOTAL"]/(Xp["AGE"]+1); return Xp

_NS={"np":np,"pd":pd,"StandardScaler":StandardScaler,"RobustScaler":RobustScaler,
     "PowerTransformer":PowerTransformer,"MinMaxScaler":MinMaxScaler,"QuantileTransformer":QuantileTransformer}
def _load(p):
    ns=dict(_NS); exec(open(p,encoding="utf-8").read(),ns); return ns["preprocess"]
FROZEN={"DeepSeek":_load("llm_raw/DeepSeek_preprocess.py"),
        "Claude Sonnet 4.6":_load("llm_raw/Claude_Sonnet_4.6_preprocess.py"),
        "GPT-5.4":_load("llm_raw/GPT-5.4_preprocess.py")}
def _caafe(frame):
    ns={"df":frame.copy(),"np":np,"pd":pd}; exec(open("llm_raw/CAAFE_code.py",encoding="utf-8").read(),ns)
    return ns["df"].select_dtypes(include=[np.number]).fillna(0)
def transform_pair(name,x_fit,x_eval):
    if name=="Manual Expert": return _manual(x_fit),_manual(x_eval)
    if name=="CAAFE":
        tr=_caafe(x_fit); te=_caafe(x_eval).reindex(columns=tr.columns,fill_value=0); return tr,te
    tr,te=FROZEN[name](x_fit.copy(),x_eval.copy())
    if not isinstance(tr,pd.DataFrame):
        cols=(list(x_fit.columns)+["CONTRIBUTION_CLAIM_RATIO","UNITS_PER_CLAIM"]) if (name=="DeepSeek" and tr.shape[1]==x_fit.shape[1]+2) else [f"f{i}" for i in range(tr.shape[1])]
        tr=pd.DataFrame(tr,index=x_fit.index,columns=cols)
    if not isinstance(te,pd.DataFrame): te=pd.DataFrame(te,index=x_eval.index,columns=tr.columns)
    return tr,te.reindex(columns=tr.columns,fill_value=0)
def san(df): return pd.DataFrame(df).replace([np.inf,-np.inf],np.nan).fillna(0.0)
def nb_se(d,k=10):
    d=np.asarray(d); n=d.size; return np.sqrt(d.var(ddof=1)*(1.0/n+(1.0/k)/(1-1.0/k)))
def tost90(d,k=10):
    d=np.asarray(d); m=d.mean(); se=nb_se(d,k); c=stats.t.ppf(0.95,d.size-1); return m,se,[m-c*se,m+c*se]
def cv_metrics(name,X,y,drop_units=False,k=10,repeats=1):
    accs,mccs,aucs=[],[],[]
    for r in range(repeats):
        for tr,va in StratifiedKFold(k,shuffle=True,random_state=(RS if repeats==1 else 2026+r)).split(X,y):
            ftr,fva=transform_pair(name,X.iloc[tr],X.iloc[va])
            if drop_units:
                uc=[c for c in ftr.columns if 'UNIT' in c.upper()]
                ftr=ftr.drop(columns=uc); fva=fva.drop(columns=[c for c in uc if c in fva.columns])
            ftr,fva=san(ftr),san(fva)
            m=LGBMClassifier(random_state=RS,verbose=-1,n_jobs=1).fit(ftr,y.iloc[tr])
            p=m.predict(fva); pr=m.predict_proba(fva)[:,1]
            accs.append(accuracy_score(y.iloc[va],p)); mccs.append(matthews_corrcoef(y.iloc[va],p)); aucs.append(roc_auc_score(y.iloc[va],pr))
    return np.array(accs),np.array(mccs),np.array(aucs)
x_tr,x_ho,y_tr,y_ho = load_and_split()
print("Setup complete |", f"{len(x_tr)+len(x_ho):,} records, {x_tr.shape[1]} predictors")

Setup complete | 24,071 records, 11 predictors


### Table 13 — model-class sensitivity (one 10-fold protocol for every column)

In [2]:
# Table 13 - model-class sensitivity.
# Every column uses the SAME single stratified 10-fold protocol (random_state=42) so the
# estimators are compared under one resampling design, exactly as the manuscript states.
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline

def lgbm_10fold(name):
    a, _, _ = cv_metrics(name, x_tr, y_tr, repeats=1)      # repeats=1 -> seed 42
    return a.mean()

def selected_C(name):
    # the manuscript reports the C chosen by one fit on the whole training partition
    p = make_pipeline(StandardScaler(),
                      LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10, 100], cv=5,
                                           max_iter=5000, random_state=RS))
    p.fit(san(transform_pair(name, x_tr, x_tr)[0]), y_tr)
    return float(p.named_steps['logisticregressioncv'].C_[0])

def lr_10fold(name, mode):
    skf = StratifiedKFold(10, shuffle=True, random_state=RS)
    accs = []
    for tr, va in skf.split(x_tr, y_tr):
        ftr, fva = transform_pair(name, x_tr.iloc[tr], x_tr.iloc[va])
        ftr, fva = san(ftr), san(fva)
        if mode == 'raw':
            est = LogisticRegression(max_iter=5000, random_state=RS)
        elif mode == 'scaled':
            est = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=5000, random_state=RS))
        else:
            est = make_pipeline(StandardScaler(),
                                LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10, 100], cv=5,
                                                     max_iter=5000, random_state=RS))
        est.fit(ftr, y_tr.iloc[tr])
        accs.append(accuracy_score(y_tr.iloc[va], est.predict(fva)))
    return float(np.mean(accs))

rows = []
for n, s in zip(NAMES, SHORT):
    lg = lgbm_10fold(n)
    raw = lr_10fold(n, 'raw')
    sca = lr_10fold(n, 'scaled')
    tun = lr_10fold(n, 'tuned')
    rows.append([s, round(lg, 4), round(raw, 4), round(sca, 4), round(tun, 4),
                 f"{selected_C(n):g}"])
t13 = pd.DataFrame(rows, columns=['Pipeline', 'LGBM', 'LR raw', 'LR scal.',
                                  'LR tuned', 'Selected C'])
span = lambda col: round(t13[col].max() - t13[col].min(), 4)
t13.loc[len(t13)] = ['Span', span('LGBM'), span('LR raw'), span('LR scal.'),
                     span('LR tuned'), '-']
t13.to_excel('results/07_table13_model_sensitivity.xlsx', index=False)
display(t13)

,Pipeline,LGBM,LR raw,LR scal.,LR tuned,Selected C
0,Manual,0.8218,0.7716,0.7892,0.7886,0.1
1,DeepSeek,0.8229,0.7841,0.7841,0.7843,0.1
2,Claude 4.6,0.8220,0.7864,0.7881,0.7879,1.0
3,GPT-5.4,0.8209,0.7885,0.7886,0.7884,10.0
4,CAAFE,0.8191,0.5863,0.6278,0.6281,10.0
5,Span,0.0038,0.2022,0.1614,0.1605,-


### Tables 9 and 10 — corrected equivalence with TOST p, and Cohen's d_z

In [3]:
# Tables 9 and 10 - corrected equivalence (with TOST p) and Cohen's d_z.
rep = {n: cv_metrics(n, x_tr, y_tr, repeats=5) for n in NAMES}   # 5 x 10-fold = 50 folds

def tost_p(d, k=10, margin=0.01):
    d = np.asarray(d); m, se = d.mean(), nb_se(d, k)
    if se == 0:
        return 0.0 if abs(m) < margin else 1.0
    df_ = d.size - 1
    return max(1 - stats.t.cdf((m + margin) / se, df_), stats.t.cdf((m - margin) / se, df_))

rows = []
for mi, mlabel in enumerate(['Accuracy', 'MCC', 'AUC']):
    for n, s in zip(NAMES[1:], SHORT[1:]):
        d = rep[NAMES[0]][mi] - rep[n][mi]
        m, se, ci = tost90(d)
        p = tost_p(d)
        rows.append([mlabel, s, round(m, 4), round(se, 4),
                     f"[{ci[0]:+.4f}, {ci[1]:+.4f}]",
                     '<0.001' if p < 0.001 else round(p, 3),
                     'Yes' if (ci[0] > -0.01 and ci[1] < 0.01) else 'No'])
t9 = pd.DataFrame(rows, columns=['Metric', 'Manual minus', 'Mean difference',
                                 'Corrected SE', 'Corrected 90% CI', 'TOST p',
                                 'Within +/-0.01'])
t9.to_excel('results/07_table9_equivalence_tost.xlsx', index=False)
display(t9)

# Table 10 - pairwise Cohen's d_z on the single 10-fold accuracies
ten = {n: cv_metrics(n, x_tr, y_tr, repeats=1)[0] for n in NAMES}
mat = pd.DataFrame(index=SHORT, columns=SHORT, dtype=object)
for i, a in enumerate(NAMES):
    for j, b in enumerate(NAMES):
        if i == j:
            mat.iloc[i, j] = '-'
        else:
            d = ten[a] - ten[b]
            mat.iloc[i, j] = round(float(d.mean() / d.std(ddof=1)), 2)
mat.to_excel('results/07_table10_cohen_dz.xlsx')
display(mat)

,Metric,Manual minus,Mean difference,Corrected SE,Corrected 90% CI,TOST p,Within +/-0.01
0,Accuracy,DeepSeek,-0.0016,0.0015,"[-0.0041, +0.0009]",<0.001,Yes
1,Accuracy,Claude 4.6,-0.0001,0.0016,"[-0.0027, +0.0025]",<0.001,Yes
2,Accuracy,GPT-5.4,-0.0002,0.0015,"[-0.0026, +0.0023]",<0.001,Yes
3,Accuracy,CAAFE,0.0010,0.0017,"[-0.0019, +0.0038]",<0.001,Yes
4,MCC,DeepSeek,-0.0031,0.0030,"[-0.0083, +0.0020]",0.015,Yes
5,MCC,Claude 4.6,0.0008,0.0033,"[-0.0047, +0.0064]",0.004,Yes
6,MCC,GPT-5.4,0.0004,0.0031,"[-0.0048, +0.0055]",0.002,Yes
7,MCC,CAAFE,0.0018,0.0035,"[-0.0041, +0.0077]",0.012,Yes
8,AUC,DeepSeek,-0.0004,0.0006,"[-0.0014, +0.0006]",<0.001,Yes
9,AUC,Claude 4.6,0.0010,0.0008,"[-0.0004, +0.0023]",<0.001,Yes


,Manual,DeepSeek,Claude 4.6,GPT-5.4,CAAFE
Manual,-,-0.41,-0.09,0.26,0.71
DeepSeek,0.41,-,0.37,0.47,0.84
Claude 4.6,0.09,-0.37,-,0.29,0.87
GPT-5.4,-0.26,-0.47,-0.29,-,0.4
CAAFE,-0.71,-0.84,-0.87,-0.4,-


### Table 15 — primary analysis repeated without UNITSTOTAL

In [4]:
# Table 15 - the no-UNITSTOTAL analysis reported fully in parallel with the main one.
repnu = {n: cv_metrics(n, x_tr, y_tr, drop_units=True, repeats=5) for n in NAMES}

def boot_ci(yt, yp, n_boot=2000, seed=2026):
    yt, yp = np.asarray(yt), np.asarray(yp)
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(yt == 1), np.flatnonzero(yt == 0)
    acc = []
    for _ in range(n_boot):
        b = np.concatenate([rng.choice(pos, pos.size, True), rng.choice(neg, neg.size, True)])
        acc.append(accuracy_score(yt[b], yp[b]))
    return float(np.percentile(acc, 2.5)), float(np.percentile(acc, 97.5))

rows = []
for n, s in zip(NAMES, SHORT):
    a, m, u = repnu[n]
    ttr, tho = transform_pair(n, x_tr, x_ho)
    uc = [c for c in ttr.columns if 'UNIT' in c.upper()]
    A = san(ttr.drop(columns=uc)); B = san(tho.drop(columns=[c for c in uc if c in tho.columns]))
    mdl = LGBMClassifier(random_state=RS, verbose=-1, n_jobs=1).fit(A, y_tr)
    pred = mdl.predict(B)
    ho = accuracy_score(y_ho, pred); lo, hi = boot_ci(y_ho, pred)
    if n == NAMES[0]:
        ci_txt, p_txt = 'reference', 'reference'
    else:
        d = repnu[NAMES[0]][0] - repnu[n][0]
        _, _, ci = tost90(d)
        p = tost_p(d)
        ci_txt = f"[{ci[0]:+.4f}, {ci[1]:+.4f}]"
        p_txt = ('<0.001' if p < 0.001 else f"{p:.3f}") + (' OK' if (ci[0] > -0.01 and ci[1] < 0.01) else ' X')
    rows.append([s, round(a.mean(), 4), round(m.mean(), 4), round(u.mean(), 4),
                 ci_txt, p_txt, round(ho, 4), f"[{lo:.4f}, {hi:.4f}]"])
t15 = pd.DataFrame(rows, columns=['Pipeline', 'CV Acc', 'CV MCC', 'CV AUC',
                                  'Corrected 90% CI (Acc)', 'TOST p', 'Holdout Acc',
                                  'Holdout 95% CI'])
t15.to_excel('results/07_table15_no_unitstotal.xlsx', index=False)
display(t15)

,Pipeline,CV Acc,CV MCC,CV AUC,Corrected 90% CI (Acc),TOST p,Holdout Acc,Holdout 95% CI
0,Manual,0.7558,0.5092,0.8387,reference,reference,0.7616,"[0.7491, 0.7734]"
1,DeepSeek,0.7556,0.5089,0.8386,"[-0.0023, +0.0029]",<0.001 OK,0.7576,"[0.7452, 0.7697]"
2,Claude 4.6,0.7556,0.5115,0.8378,"[-0.0032, +0.0037]",<0.001 OK,0.7599,"[0.7479, 0.7718]"
3,GPT-5.4,0.7551,0.5064,0.8379,"[-0.0022, +0.0037]",<0.001 OK,0.7533,"[0.7416, 0.7651]"
4,CAAFE,0.7569,0.5117,0.8391,"[-0.0037, +0.0015]",<0.001 OK,0.7585,"[0.7462, 0.7703]"


### Table 7 — hyperparameters in force and Logistic-Regression convergence

In [5]:
# Table 7 - every hyperparameter actually in force, plus Logistic-Regression convergence.
from sklearn.exceptions import ConvergenceWarning
import warnings as _w

probe = LGBMClassifier(random_state=RS, verbose=-1, n_jobs=1)
keep = ['boosting_type', 'n_estimators', 'learning_rate', 'num_leaves', 'max_depth',
        'min_child_samples', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda',
        'random_state']
lgbm_params = ', '.join(f"{k}={probe.get_params()[k]}" for k in keep)

conv = {}
for n in NAMES:
    ttr, _ = transform_pair(n, x_tr, x_ho)
    A = san(ttr)
    for cfg, mk in (('raw', lambda: LogisticRegression(max_iter=5000, random_state=RS)),
                    ('scaled', lambda: make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=5000, random_state=RS))),
                    ('tuned', lambda: make_pipeline(StandardScaler(),
                        LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10, 100], cv=5,
                                             max_iter=5000, random_state=RS)))):
        with _w.catch_warnings(record=True) as w:
            _w.simplefilter('always', ConvergenceWarning)
            est = mk(); est.fit(A, y_tr)
            warned = any(issubclass(x.category, ConvergenceWarning) for x in w)
        lr = est[-1] if hasattr(est, '__getitem__') else est
        conv.setdefault(cfg, []).append((int(np.max(lr.n_iter_)), not warned))

t7 = pd.DataFrame([
    ['LightGBM (all analyses)', lgbm_params + ' (library defaults, no tuning)', 'n/a'],
    ['LR as delivered', 'solver=lbfgs, penalty=L2, C=1.0, max_iter=5000, no scaling',
     f"converged; max {max(i for i, _ in conv['raw'])} iterations"],
    ['LR uniformly scaled', 'StandardScaler + lbfgs, L2, C=1.0, max_iter=5000',
     f"converged; max {max(i for i, _ in conv['scaled'])} iterations"],
    ['LR scaled + nested-tuned',
     'StandardScaler + LogisticRegressionCV, C in {0.01,0.1,1,10,100}, 5-fold inner CV',
     f"converged; max {max(i for i, _ in conv['tuned'])} iterations"],
], columns=['Estimator configuration', 'Hyperparameters in force', 'Convergence'])
t7.to_excel('results/07_table7_estimator_config.xlsx', index=False)
display(t7)

,Estimator configuration,Hyperparameters in force,Convergence
0,LightGBM (all analyses),"boosting_type=gbdt, n_estimators=100, learning...",n/a
1,LR as delivered,"solver=lbfgs, penalty=L2, C=1.0, max_iter=5000...",converged; max 642 iterations
2,LR uniformly scaled,"StandardScaler + lbfgs, L2, C=1.0, max_iter=5000",converged; max 88 iterations
3,LR scaled + nested-tuned,"StandardScaler + LogisticRegressionCV, C in {0...",converged; max 75 iterations
